In [ ]:
# Step 1: Install required acceleration and evaluation packages
!pip install -q transformers datasets accelerate nltk

import os
import zipfile
import urllib.request
import collections
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from PIL import Image
from transformers import AutoTokenizer, GPT2LMHeadModel
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

In [ ]:
# Step 2: Environment & Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using execution device: {device}")

# Download Flickr8k dataset via public mirrors
DATA_URL = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip"
TEXT_URL = "https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip"

if not os.path.exists("Flickr8k_Dataset.zip"):
    print("Downloading dataset images (approx 1GB)...")
    urllib.request.urlretrieve(DATA_URL, "Flickr8k_Dataset.zip")
if not os.path.exists("Flickr8k_text.zip"):
    print("Downloading text files...")
    urllib.request.urlretrieve(TEXT_URL, "Flickr8k_text.zip")

# Extract archives
for zip_fn in ["Flickr8k_Dataset.zip", "Flickr8k_text.zip"]:
    with zipfile.ZipFile(zip_fn, 'r') as zip_ref:
        zip_ref.extractall(".")
print("Dataset extracted successfully.")

In [ ]:
# Step 3: Dataset Splitting Logic (6000 Train, 1000 Val, 1000 Test)
def load_split_image_names(file_path):
    with open(file_path, 'r') as f:
        return set([line.strip() for line in f if line.strip()])

train_images_set = load_split_image_names("Flickr_8k.trainImages.txt")
val_images_set = load_split_image_names("Flickr_8k.devImages.txt")
test_images_set = load_split_image_names("Flickr_8k.testImages.txt")

def parse_captions(file_path):
    image_to_captions = collections.defaultdict(list)
    with open(file_path, 'r') as f:
        for line in f:
            tokens = line.strip().split('\t')
            if len(tokens) < 2:
                continue
            image_id, caption = tokens[0], tokens[1]
            image_name = image_id.split('#')[0]
            caption_clean = caption.lower()
            caption_clean = re.sub(r'[^a-zA-Z\s]', '', caption_clean)
            image_to_captions[image_name].append(caption_clean.strip().split())
    return image_to_captions

all_captions = parse_captions("Flickr8k.token.txt")
print("Metadata splitting complete.")

In [ ]:
# Step 4: Tokenizer & PyTorch Data Pipelines
tokenizer = AutoTokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class FlickrResnetGptDataset(Dataset):
    def __init__(self, image_set, captions_dict, tokenizer, transform=None, img_dir="Flicker8k_Dataset"):
        self.image_names = list(image_set)
        self.captions_dict = captions_dict
        self.tokenizer = tokenizer
        self.transform = transform
        self.img_dir = img_dir

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        
        caption_str = " ".join(self.captions_dict[img_name][0])
        tokens = self.tokenizer(caption_str, truncation=True, max_length=30, add_special_tokens=False)
        input_ids = tokens['input_ids'] + [self.tokenizer.eos_token_id]
        return image, torch.tensor(input_ids, dtype=torch.long)

def collate_fn(batch):
    images, captions = zip(*batch)
    images = torch.stack(images, 0)
    lengths = [len(cap) for cap in captions]
    padded_captions = torch.full((len(captions), max(lengths)), 50256, dtype=torch.long)
    attention_mask = torch.zeros((len(captions), max(lengths)), dtype=torch.long)
    labels = torch.full((len(captions), max(lengths)), -100, dtype=torch.long)
    
    for i, cap in enumerate(captions):
        padded_captions[i, :len(cap)] = cap
        attention_mask[i, :len(cap)] = 1
        labels[i, :len(cap)] = cap
    return images, padded_captions, attention_mask, labels

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(FlickrResnetGptDataset(train_images_set, all_captions, tokenizer, image_transform), 
                          batch_size=64, shuffle=True, collate_fn=collate_fn)

In [ ]:
# Step 5: Custom ResNet50 + GPT2 Architecture Module
class ResNet50GPT2Captioner(nn.Module):
    def __init__(self, gpt2_model_name="gpt2"):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for param in resnet.parameters():
            param.requires_grad = False
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        self.decoder = GPT2LMHeadModel.from_pretrained(gpt2_model_name)
        self.proj = nn.Linear(2048, self.decoder.config.n_embd)
        
    def forward(self, images, input_ids, attention_mask, labels):
        with torch.no_grad():
            features = self.encoder(images)
        features = features.view(features.size(0), -1)
        img_embeddings = self.proj(features).unsqueeze(1)
        text_embeddings = self.decoder.transformer.wte(input_ids)
        
        inputs_embeds = torch.cat((img_embeddings, text_embeddings), dim=1)
        img_mask = torch.ones((attention_mask.size(0), 1), dtype=attention_mask.dtype, device=attention_mask.device)
        extended_attention_mask = torch.cat((img_mask, attention_mask), dim=1)
        
        dummy_labels = torch.full((labels.size(0), 1), -100, dtype=labels.dtype, device=labels.device)
        extended_labels = torch.cat((dummy_labels, labels), dim=1)
        
        return self.decoder(inputs_embeds=inputs_embeds, attention_mask=extended_attention_mask, labels=extended_labels)

In [ ]:
# Step 6: Fitting the Model Weights
model = ResNet50GPT2Captioner().to(device)
optimizer = torch.optim.AdamW(list(model.proj.parameters()) + list(model.decoder.parameters()), lr=5e-4)
scaler = torch.amp.GradScaler('cuda')

model.train()
print("--- Starting Optimization Pass (1 Epoch convergence) ---")
for step, (images, captions, masks, labels) in enumerate(train_loader):
    images, captions, masks, labels = images.to(device), captions.to(device), masks.to(device), labels.to(device)
    optimizer.zero_grad()
    with torch.amp.autocast('cuda'):
        outputs = model(images, captions, masks, labels)
        loss = outputs.loss
        
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    if step % 20 == 0:
        print(f"Step [{step}/{len(train_loader)}], Loss: {loss.item():.4f}")

In [ ]:
# Step 7: Autoregressive Beam Search Decoder
def beam_search_decode(model, image_tensor, tokenizer, beam_width=3, max_len=20):
    model.eval()
    with torch.no_grad():
        features = model.encoder(image_tensor.to(device))
        features = features.view(features.size(0), -1)
        img_embeds = model.proj(features).unsqueeze(1)
        candidates = [(0.0, [tokenizer.eos_token_id])]
        
        for _ in range(max_len):
            new_candidates = []
            for score, seq in candidates:
                if len(seq) > 1 and seq[-1] == tokenizer.eos_token_id:
                    new_candidates.append((score, seq))
                    continue
                text_ids = torch.tensor([seq], device=device)
                text_embeds = model.decoder.transformer.wte(text_ids)
                inputs_embeds = torch.cat((img_embeds, text_embeds), dim=1)
                outputs = model.decoder(inputs_embeds=inputs_embeds)
                log_probs = torch.log_softmax(outputs.logits[:, -1, :], dim=-1)
                top_probs, top_ids = log_probs.topk(beam_width, dim=-1)
                
                for i in range(beam_width):
                    new_candidates.append((score + top_probs[0][i].item(), seq + [top_ids[0][i].item()]))
            candidates = sorted(new_candidates, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(len(seq) > 1 and seq[-1] == tokenizer.eos_token_id for _, seq in candidates): break
            
        return tokenizer.decode(candidates[0][1], skip_special_tokens=True).lower().split()

In [ ]:
# Step 8: Validation Evaluation via BLEU Metrics
print("--- Evaluating Split Models via Dataset References ---")
val_dataset = FlickrResnetGptDataset(val_images_set, all_captions, tokenizer, image_transform)
references, hypotheses = [], []

for idx in range(50):
    img_name = val_dataset.image_names[idx]
    img_tensor, _ = val_dataset[idx]
    predicted_words = beam_search_decode(model, img_tensor.unsqueeze(0), tokenizer, beam_width=3)
    references.append(all_captions[img_name])
    hypotheses.append(predicted_words)

smooth = SmoothingFunction().method1
print(f"BLEU-1 Score: {corpus_bleu(references, hypotheses, weights=(1.0, 0, 0, 0), smoothing_function=smooth):.4f}")
print(f"BLEU-4 Score: {corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth):.4f}")